# 10 — Human-in-the-Loop Agents

## Background

Fully autonomous agents are dangerous when task stakes are high or agent confidence is low. Human-in-the-loop (HITL) design inserts approval gates at the right decision points — not everywhere (which destroys autonomy) and not nowhere (which destroys safety).

The research community distinguishes three modes: *approval-gated* (explicit human sign-off), *interrupt-on-uncertainty* (agent escalates when it doesn't know), and *asynchronous oversight* (human reviews after the fact with rollback). Real systems mix all three.

## What You'll Learn

- Confidence-based escalation: when should an agent stop and ask?
- Approval gate patterns: synchronous blocking vs. async queue
- Rollback design: how to make actions reversible before approval
- Timeout handling: what to do when the human doesn't respond
- Escalation cost modeling: trading autonomy for safety

## Why This Matters

HITL is not a bolt-on — it's an architectural decision. Adding it after the fact means retrofitting every action to be reversible and every decision point to expose a confidence score. Getting the design right early keeps agents deployable in regulated and high-stakes environments.

In [ ]:
import time, uuid, random
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional
from enum import Enum
import numpy as np

class EscalationReason(str, Enum):
    LOW_CONFIDENCE    = 'low_confidence'
    HIGH_COST_ACTION  = 'high_cost_action'
    IRREVERSIBLE      = 'irreversible'
    POLICY_REQUIRED   = 'policy_required'
    AMBIGUOUS_INTENT  = 'ambiguous_intent'

class ApprovalStatus(str, Enum):
    PENDING  = 'pending'
    APPROVED = 'approved'
    REJECTED = 'rejected'
    TIMEOUT  = 'timeout'

@dataclass
class ApprovalRequest:
    request_id:   str
    action:       str
    context:      Dict[str, Any]
    reason:       EscalationReason
    confidence:   float
    created_at:   float = field(default_factory=time.time)
    timeout_s:    float = 30.0
    status:       ApprovalStatus = ApprovalStatus.PENDING
    resolved_at:  Optional[float] = None
    reviewer_note: Optional[str] = None

    def is_expired(self) -> bool:
        return (time.time() - self.created_at) > self.timeout_s

    def resolve(self, status: ApprovalStatus, note: str = ''):
        self.status      = status
        self.resolved_at = time.time()
        self.reviewer_note = note

print('HITL data classes defined.')


## Confidence Estimator

Before each action, the agent computes a confidence score. Scores below a threshold trigger escalation. The estimator uses multiple signals: semantic certainty, action reversibility, and policy flags.

In [ ]:
@dataclass
class ActionDescriptor:
    name:          str
    description:   str
    reversible:    bool = True
    cost_level:    int  = 1        # 1=low, 2=medium, 3=high
    policy_gate:   bool = False    # always requires human approval

TOOL_REGISTRY: Dict[str, ActionDescriptor] = {
    'read_file':       ActionDescriptor('read_file',       'Read a file from disk',           reversible=True,  cost_level=1),
    'write_file':      ActionDescriptor('write_file',      'Write/overwrite a file',          reversible=False, cost_level=2),
    'delete_file':     ActionDescriptor('delete_file',     'Permanently delete a file',       reversible=False, cost_level=3),
    'send_email':      ActionDescriptor('send_email',      'Send an email to a recipient',    reversible=False, cost_level=2, policy_gate=True),
    'run_query':       ActionDescriptor('run_query',       'Run a read-only DB query',        reversible=True,  cost_level=1),
    'run_mutation':    ActionDescriptor('run_mutation',    'Run a DB write/update',           reversible=False, cost_level=3, policy_gate=True),
    'call_api':        ActionDescriptor('call_api',        'Call an external API',            reversible=True,  cost_level=2),
    'deploy_artifact': ActionDescriptor('deploy_artifact', 'Deploy to production environment',reversible=False, cost_level=3, policy_gate=True),
}

class ConfidenceEstimator:
    def __init__(self, low_threshold: float = 0.6, medium_threshold: float = 0.8):
        self.low_threshold    = low_threshold
        self.medium_threshold = medium_threshold

    def estimate(self, action: str, semantic_score: float) -> tuple:
        desc = TOOL_REGISTRY.get(action)
        if desc is None:
            return 0.3, EscalationReason.AMBIGUOUS_INTENT

        # Penalise for cost and irreversibility
        cost_penalty = (desc.cost_level - 1) * 0.1
        rev_bonus    = 0.1 if desc.reversible else -0.1
        confidence   = max(0.0, min(1.0, semantic_score - cost_penalty + rev_bonus))

        if desc.policy_gate:
            return confidence, EscalationReason.POLICY_REQUIRED
        if not desc.reversible and confidence < self.medium_threshold:
            return confidence, EscalationReason.IRREVERSIBLE
        if desc.cost_level >= 3:
            return confidence, EscalationReason.HIGH_COST_ACTION
        if confidence < self.low_threshold:
            return confidence, EscalationReason.LOW_CONFIDENCE

        return confidence, None   # None = no escalation needed

est = ConfidenceEstimator()
test_cases = [
    ('read_file',       0.95),
    ('write_file',      0.85),
    ('delete_file',     0.90),
    ('send_email',      0.92),
    ('run_query',       0.70),
    ('deploy_artifact', 0.88),
    ('unknown_tool',    0.70),
]
print(f'  {"Action":<20} {"Semantic":>9} {"Confidence":>11} {"Escalation Reason"}')
for action, sem in test_cases:
    conf, reason = est.estimate(action, sem)
    r = reason.value if reason else 'none'
    print(f'  {action:<20} {sem:>9.2f} {conf:>11.3f} {r}')


## Approval Gate & Queue

The approval queue holds pending requests. In production this would be backed by a database and push notifications to reviewers. Here we simulate synchronous and asynchronous resolution paths.

In [ ]:
class ApprovalQueue:
    def __init__(self):
        self._queue: Dict[str, ApprovalRequest] = {}
        self._resolved: List[ApprovalRequest]   = []

    def submit(self, action: str, context: Dict, reason: EscalationReason,
               confidence: float, timeout_s: float = 30.0) -> ApprovalRequest:
        req = ApprovalRequest(
            request_id = str(uuid.uuid4()),
            action     = action,
            context    = context,
            reason     = reason,
            confidence = confidence,
            timeout_s  = timeout_s,
        )
        self._queue[req.request_id] = req
        return req

    def resolve(self, request_id: str, approved: bool, note: str = '') -> bool:
        req = self._queue.get(request_id)
        if req is None: return False
        status = ApprovalStatus.APPROVED if approved else ApprovalStatus.REJECTED
        req.resolve(status, note)
        self._resolved.append(req)
        del self._queue[request_id]
        return True

    def expire_old(self):
        to_expire = [rid for rid, req in self._queue.items() if req.is_expired()]
        for rid in to_expire:
            self._queue[rid].resolve(ApprovalStatus.TIMEOUT, 'auto-expired')
            self._resolved.append(self._queue.pop(rid))

    def pending(self) -> List[ApprovalRequest]:
        return list(self._queue.values())

    def stats(self) -> Dict:
        approved  = sum(1 for r in self._resolved if r.status == ApprovalStatus.APPROVED)
        rejected  = sum(1 for r in self._resolved if r.status == ApprovalStatus.REJECTED)
        timed_out = sum(1 for r in self._resolved if r.status == ApprovalStatus.TIMEOUT)
        return {'pending': len(self._queue), 'approved': approved,
                'rejected': rejected, 'timed_out': timed_out,
                'total_resolved': len(self._resolved)}

print('ApprovalQueue defined.')


## HITL Agent

The agent wraps every action call with confidence estimation. If escalation is needed, it submits to the queue and either blocks for approval or falls back gracefully on timeout.

In [ ]:
@dataclass
class AgentAction:
    action:  str
    args:    Dict[str, Any]
    result:  Optional[Any] = None
    skipped: bool = False
    reason:  str  = ''

class HITLAgent:
    def __init__(self, auto_approve_threshold: float = 0.85,
                 approval_timeout_s: float = 5.0):
        self.estimator           = ConfidenceEstimator()
        self.queue               = ApprovalQueue()
        self.auto_threshold      = auto_approve_threshold
        self.approval_timeout_s  = approval_timeout_s
        self.history:            List[AgentAction] = []

    def _execute_tool(self, action: str, args: Dict) -> str:
        # Stub: real impl calls actual tools
        return f'[RESULT of {action}({args})]'

    def _simulate_human_review(self, req: ApprovalRequest) -> bool:
        # Simulate a human approving high-confidence requests and rejecting low ones
        time.sleep(0.01)  # simulated review latency
        if req.reason == EscalationReason.POLICY_REQUIRED and req.confidence > 0.85:
            return True
        if req.reason == EscalationReason.IRREVERSIBLE and req.confidence > 0.75:
            return True
        if req.reason == EscalationReason.HIGH_COST_ACTION and req.confidence > 0.80:
            return True
        return False

    def act(self, action: str, args: Dict, semantic_score: float = 0.85) -> AgentAction:
        agent_action = AgentAction(action=action, args=args)
        confidence, esc_reason = self.estimator.estimate(action, semantic_score)

        if esc_reason is None:
            # Auto-execute
            agent_action.result = self._execute_tool(action, args)
            agent_action.reason = 'auto'
        else:
            # Submit for approval
            req = self.queue.submit(action, args, esc_reason, confidence,
                                   timeout_s=self.approval_timeout_s)
            approved = self._simulate_human_review(req)
            self.queue.resolve(req.request_id, approved,
                               note='simulated reviewer' if approved else 'reviewer rejected')
            if approved:
                agent_action.result  = self._execute_tool(action, args)
                agent_action.reason  = f'approved ({esc_reason.value})'
            else:
                agent_action.skipped = True
                agent_action.reason  = f'rejected ({esc_reason.value})'

        self.history.append(agent_action)
        return agent_action

agent = HITLAgent(approval_timeout_s=5.0)

task_steps = [
    ('read_file',       {'path': '/data/report.csv'},               0.95),
    ('run_query',       {'sql': 'SELECT * FROM users LIMIT 10'},    0.88),
    ('write_file',      {'path': '/out/summary.txt', 'data': '...'}, 0.82),
    ('delete_file',     {'path': '/tmp/old.log'},                   0.91),
    ('send_email',      {'to': 'boss@co.com', 'body': 'Report!'},   0.93),
    ('deploy_artifact', {'artifact': 'v1.2.3.tar.gz'},              0.88),
    ('run_mutation',    {'sql': 'UPDATE users SET active=0'},        0.65),
]

print(f'  {"Action":<20} {"Reason":<30} {"Skipped"}')
for action, args, sem in task_steps:
    a = agent.act(action, args, semantic_score=sem)
    print(f'  {action:<20} {a.reason:<30} {a.skipped}')

print(f'\nApproval queue stats: {agent.queue.stats()}')


## Escalation Cost Analysis

Model the tradeoff: each escalation has a latency cost (human review time) and a benefit (error rate reduction). Find the threshold that minimises total expected cost.

In [ ]:
import numpy as np

# Parameters
human_review_latency_s = 30.0    # avg time for human to review
auto_error_rate_at_1   = 0.25    # error rate at confidence=1.0 (agent makes mistakes)
auto_error_rate_at_0   = 0.85    # error rate at confidence=0.0
error_cost_usd         = 50.0    # cost per downstream error
review_cost_usd        = 2.0     # cost per human review (time * hourly rate)

thresholds = np.linspace(0, 1, 100)

# Simulate: at each threshold, some fraction of tasks escalate
# Tasks below threshold escalate, tasks above auto-execute
# Assume confidence scores ~ Beta(5, 2) — skewed toward high confidence
rng = np.random.default_rng(42)
n_tasks = 10000
confidences = rng.beta(5, 2, n_tasks)

total_costs = []
for thresh in thresholds:
    auto_mask   = confidences >= thresh
    review_mask = ~auto_mask
    n_auto      = auto_mask.sum()
    n_review    = review_mask.sum()

    # Auto-executed tasks: error rate decreases with confidence
    auto_confs  = confidences[auto_mask]
    auto_errors = ((1 - auto_confs) * (auto_error_rate_at_0 - auto_error_rate_at_1)
                   + auto_error_rate_at_1).sum()

    # Reviewed tasks: human catches most errors, assume 5% slip-through
    review_slip = n_review * 0.05

    cost = (auto_errors + review_slip) * error_cost_usd + n_review * review_cost_usd
    total_costs.append(cost)

best_idx   = int(np.argmin(total_costs))
best_thresh = thresholds[best_idx]

print('Escalation cost analysis:')
print(f'  Optimal threshold : {best_thresh:.3f}')
print(f'  Min total cost    : ${total_costs[best_idx]:.2f} (over {n_tasks} tasks)')
print(f'  At threshold {best_thresh:.2f}:')
auto_frac   = (confidences >= best_thresh).mean()
review_frac = 1 - auto_frac
print(f'    Auto-executed   : {auto_frac*100:.1f}%')
print(f'    Escalated       : {review_frac*100:.1f}%')
print(f'  At threshold 0.00 (always escalate): ${total_costs[0]:.2f}')
print(f'  At threshold 1.00 (never escalate) : ${total_costs[-1]:.2f}')


## Key Takeaways

- Confidence estimation must account for action cost, reversibility, and policy flags — not just model uncertainty
- Approval gates should be async by default; synchronous blocking is only acceptable when the action cannot proceed without a decision
- The optimal escalation threshold depends on error cost vs. review cost — model it explicitly rather than choosing arbitrarily
- Every irreversible action needs either human approval or a rollback mechanism — never both optional
- Timeout handling is not edge-case — it's the dominant failure mode in human-in-the-loop systems